# Molecular and reaction representations

This notebook computes the feature matrices used in the modelling notebook. Each representation is written to a separate CSV file; rows follow the order of the input dataset and are keyed by `reaction_id`. Targets, groups and other metadata are not written to the feature files; the modelling notebook reads them from the dataset copy produced here.

| File | Representation | Columns per reaction |
|---|---|---|
| `features_physchem.csv` | RDKit 2D descriptors (all except `Ipc`), per role | 3 × 216 + 2 |
| `features_maccs.csv` | MACCS structural keys (bit 0 omitted), per role | 3 × 166 + 2 |
| `features_ecfp4.csv` | Morgan fingerprint, radius 2, binary, chirality-aware, per role | 3 × 2048 + 2 |
| `features_ecfp4_count.csv` | Morgan fingerprint, radius 2, counts, chirality-aware, per role | 3 × 2048 + 2 |
| `features_drfp.csv` | Differential reaction fingerprint of `electrophile.nucleophile>catalyst>product`, plus catalyst axial-configuration flags | 2048 + 2 |
| `features_onehot.csv` | One-hot identity encoding of catalyst (SMILES + axial configuration), electrophile and nucleophile; column legend in `onehot_legend.csv` | n_cat + n_ele + n_nuc |
| `features_dft.csv` | DFT descriptors per role, mapped onto reactions (written only if per-molecule DFT tables are supplied, Section 7) | 3 × n_desc + 2 |
| `features_conditions.csv` | Temperature, catalyst loading, solvent Kamlet–Taft α, β, π\* and dielectric constant ε | 6 |

Roles: organocatalyst (`cat_`), electrophile (`ele_`), nucleophile (`nuc_`). A catalyst is identified by its SMILES together with its axial configuration (`axial_configuration`): SMILES cannot encode axial chirality, so atropisomeric catalysts that differ only in the configuration of the biaryl axis share one SMILES string. Two binary columns, `cat_axial_R` and `cat_axial_S` (both 0 for catalysts without a stereogenic axis), are appended to the catalyst block of every representation. Values are written without filtering or scaling; feature selection and standardisation are performed inside cross-validation folds in the modelling notebook.

Edit the configuration cell only.

## 1. Configuration

In [ ]:
from pathlib import Path

# Paths are relative to notebooks/ (run the notebook from that folder).
DATA_PATH = Path('../data/processed/mannich_dataset_hbd.csv')   # input dataset
OUT_DIR   = Path('../data/representations')                   # output folder

ROLE_COLS   = {'cat': 'organocatalyst', 'ele': 'electrophile', 'nuc': 'nucleophile'}
PRODUCT_COL = 'product'
AXIAL_COL   = 'axial_configuration'                 # 'R', 'S' or 'none'
TEMP_COL    = 'temperature'                          # °C
LOADING_COL = 'organocatalyst_concentration(mol%)'
SOLVENT_COL = 'solvent'

# Optional external solvent table (CSV with columns: solvent, alpha, beta, pi_star, eps, source).
# If None, the built-in table in Section 8 is used.
SOLVENT_TABLE_PATH = None

# Optional per-molecule DFT descriptor tables (Section 7). One CSV per role, or None to skip.
# Each table: a 'smiles' column (catalyst table also 'axial_configuration': R/S/none) + numeric descriptor columns.
DFT_FILES = {'cat': None, 'ele': None, 'nuc': None}

# Fingerprint parameters
MORGAN_RADIUS      = 2        # radius 2 = ECFP4
MORGAN_FP_SIZE     = 2048
MORGAN_CHIRALITY   = True
DRFP_FP_SIZE       = 2048
DRFP_RADIUS        = 3        # DRFP default (DRFP6)

## 2. Environment and input

In [ ]:
import hashlib, json, platform, importlib.util
import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator

if importlib.util.find_spec('drfp') is None:
    raise ImportError('Package "drfp" is required: pip install drfp')
import drfp
from drfp import DrfpEncoder

RDLogger.DisableLog('rdApp.*')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

df = pd.read_csv(DATA_PATH, float_precision='round_trip')   # exact float round-trip
required = list(ROLE_COLS.values()) + [PRODUCT_COL, AXIAL_COL, TEMP_COL, LOADING_COL, SOLVENT_COL]
missing = [c for c in required if c not in df.columns]
assert not missing, f'Missing columns: {missing}'
assert 'reaction_id' not in df.columns or df['reaction_id'].is_unique, 'Existing reaction_id is not unique'

for col in list(ROLE_COLS.values()) + [PRODUCT_COL]:
    bad = df[col].map(lambda s: not isinstance(s, str) or Chem.MolFromSmiles(s) is None)
    assert not bad.any(), f'{bad.sum()} unparsable SMILES in "{col}" (rows {list(df.index[bad][:10])})'

axial = df[AXIAL_COL].fillna('none').astype(str).str.strip()
bad_axial = sorted(set(axial) - {'R', 'S', 'none'})
assert not bad_axial, f'Unexpected values in "{AXIAL_COL}": {bad_axial}'
df[AXIAL_COL] = axial
print(f'{len(df)} reactions, input SHA-256 {sha256(DATA_PATH)[:16]}…')

## 3. Reaction identifiers and catalyst-aware reaction SMILES

`reaction_id` is the zero-based row position in the input file. `reaction_smiles_with_ocat` places the organocatalyst in the agent position, `electrophile.nucleophile>catalyst>product`.

In [ ]:
df = df.reset_index(drop=True)
if 'reaction_id' not in df.columns:
    df.insert(0, 'reaction_id', np.arange(len(df)))
df['reaction_smiles_with_ocat'] = (df[ROLE_COLS['ele']] + '.' + df[ROLE_COLS['nuc']] + '>'
                                   + df[ROLE_COLS['cat']] + '>' + df[PRODUCT_COL])
ids = df[['reaction_id']]
df[['reaction_id', 'reaction_smiles_with_ocat']].head(3)

## 4. Per-molecule representations

Each unique SMILES of each role is featurised once; vectors are then mapped onto reactions and concatenated in the fixed order catalyst, electrophile, nucleophile. Catalyst vectors are indexed by the pair (SMILES, axial configuration) and carry the two axial flags. Input SMILES are parsed as given; no additional standardisation is applied (the catalyst column is already desalted).

* **Physicochemical descriptors**: all RDKit 2D descriptors in `Descriptors._descList` except `Ipc`, whose value grows exponentially with molecular size and overflows double precision (`AvgIpc` is retained). Non-finite values are stored as missing.
* **MACCS keys**: RDKit implementation, 167 bits; bit 0 is unused by definition and omitted.
* **ECFP4**: Morgan fingerprint, radius 2, 2048 bits, `includeChirality=True` (atom chirality tags enter the atom invariants).
* **Count ECFP4**: identical generator, with the number of occurrences of each hashed environment instead of presence.

In [ ]:
DESC_NAMES = [n for n, _ in Descriptors._descList if n != 'Ipc']
DESC_FUNCS = dict(Descriptors._descList)
MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_FP_SIZE,
                                                        includeChirality=MORGAN_CHIRALITY)

def physchem(mol):
    out = np.empty(len(DESC_NAMES))
    for i, name in enumerate(DESC_NAMES):
        try:
            v = float(DESC_FUNCS[name](mol))
        except Exception:
            v = np.nan
        out[i] = v if np.isfinite(v) else np.nan
    return out

def maccs(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array([fp.GetBit(i) for i in range(1, fp.GetNumBits())], dtype=np.uint8)

def ecfp4(mol):
    return MORGAN_GEN.GetFingerprintAsNumPy(mol).astype(np.uint8)

def ecfp4_count(mol):
    return MORGAN_GEN.GetCountFingerprintAsNumPy(mol).astype(np.uint32)

MOL_REPRESENTATIONS = {
    'physchem':    (physchem,    DESC_NAMES),
    'maccs':       (maccs,       [f'maccs_{i:03d}' for i in range(1, 167)]),
    'ecfp4':       (ecfp4,       [f'ecfp4_{i:04d}' for i in range(MORGAN_FP_SIZE)]),
    'ecfp4_count': (ecfp4_count, [f'ecfp4c_{i:04d}' for i in range(MORGAN_FP_SIZE)]),
}

unique_mols = {role: pd.unique(df[col]) for role, col in ROLE_COLS.items()}
mol_objects = {s: Chem.MolFromSmiles(s) for role in unique_mols for s in unique_mols[role]}

cat_keys = df[[ROLE_COLS['cat'], AXIAL_COL]].drop_duplicates().reset_index(drop=True)
cat_keys.index = cat_keys[ROLE_COLS['cat']] + '|' + cat_keys[AXIAL_COL]
df_cat_key = df[ROLE_COLS['cat']] + '|' + df[AXIAL_COL]
axial_flags = pd.DataFrame({'cat_axial_R': (cat_keys[AXIAL_COL] == 'R').astype(np.uint8),
                            'cat_axial_S': (cat_keys[AXIAL_COL] == 'S').astype(np.uint8)}, index=cat_keys.index)
print({role: len(v) for role, v in unique_mols.items()},
      f'| catalysts as (SMILES, axial configuration): {len(cat_keys)}')

In [ ]:
per_molecule = {}   # (representation, role) -> DataFrame indexed by SMILES
features = {}
for rep, (fn, names) in MOL_REPRESENTATIONS.items():
    blocks = []
    for role, col in ROLE_COLS.items():
        smiles = unique_mols[role]
        lut = pd.DataFrame(np.vstack([fn(mol_objects[s]) for s in smiles]), index=smiles,
                           columns=[f'{role}_{n}' for n in names])
        if role == 'cat':
            lut = pd.concat([lut.loc[cat_keys[col]].set_axis(cat_keys.index), axial_flags], axis=1)
            keys = df_cat_key
        else:
            keys = df[col]
        per_molecule[(rep, role)] = lut
        blocks.append(lut.loc[keys].reset_index(drop=True))
    features[rep] = pd.concat([ids] + blocks, axis=1)
    print(f'{rep:12s} {features[rep].shape[1] - 1:6d} columns')

## 5. Reaction-level representation: DRFP

The differential reaction fingerprint (Probst, Schwaller, Reymond, *Digital Discovery* **2022**, 1, 91) encodes the symmetric difference between the sets of circular substructures (radius 3, rings included) of the left- and right-hand sides of a reaction, hashed and folded to 2048 bits. In the `drfp` implementation, agents are appended to the reactants; the catalyst, being absent from the product, therefore contributes its substructures to the symmetric difference except where they coincide with substructures of the product. Because the reaction SMILES cannot carry the axial configuration of the catalyst, the two axial flags are appended. Note that DRFP is the only representation in this notebook that uses the product structure.

In [ ]:
drfp_matrix = np.vstack(DrfpEncoder.encode(df['reaction_smiles_with_ocat'].tolist(),
                                            n_folded_length=DRFP_FP_SIZE, radius=DRFP_RADIUS))
features['drfp'] = pd.concat([ids, pd.DataFrame(drfp_matrix.astype(np.uint8),
                                                columns=[f'drfp_{i:04d}' for i in range(DRFP_FP_SIZE)]),
                              axial_flags.loc[df_cat_key].reset_index(drop=True)], axis=1)
print(f"drfp         {features['drfp'].shape[1] - 1:6d} columns")

## 6. Identity encoding: one-hot

Each unique catalyst (SMILES with axial configuration), electrophile and nucleophile receives one binary column; a reaction has exactly three non-zero entries. The encoding carries no structural information and serves as a reference for how much of the performance of structure-based representations can be obtained from component identity alone. A component that is absent from the training fold is represented by an all-zero block, so under leave-one-catalyst-out and leave-one-group-out validation this representation can only use the remaining components. The mapping from columns to structures (and to the identifiers of the input file, where present) is written to `onehot_legend.csv`.

In [ ]:
ID_COLS = {'cat': 'catalyst_id', 'ele': 'electrophile_id', 'nuc': 'nucleophile_id'}

ohe_blocks, legend = [], []
for role, col in ROLE_COLS.items():
    if role == 'cat':
        keys, row_keys = list(cat_keys.index), df_cat_key
    else:
        keys, row_keys = list(unique_mols[role]), df[col]
    names = [f'{role}_ohe_{i:03d}' for i in range(len(keys))]
    lut = pd.DataFrame(np.eye(len(keys), dtype=np.uint8), index=keys, columns=names)
    per_molecule[('onehot', role)] = lut
    ohe_blocks.append(lut.loc[row_keys].reset_index(drop=True))
    first = df.assign(_key=row_keys.values).drop_duplicates('_key').set_index('_key')
    for name, key in zip(names, keys):
        legend.append({'column': name, 'role': role, 'smiles': first.at[key, col],
                       'axial_configuration': first.at[key, AXIAL_COL] if role == 'cat' else '',
                       'source_id': first.at[key, ID_COLS[role]] if ID_COLS[role] in df.columns else ''})
features['onehot'] = pd.concat([ids] + ohe_blocks, axis=1)
onehot_legend = pd.DataFrame(legend)
onehot_legend.to_csv(OUT_DIR / 'onehot_legend.csv', index=False)
assert (features['onehot'].drop(columns='reaction_id').sum(axis=1) == 3).all()
print(f"onehot       {features['onehot'].shape[1] - 1:6d} columns",
      {r: int((onehot_legend.role == r).sum()) for r in ROLE_COLS})

## 7. DFT descriptors (optional)

DFT descriptors are computed outside this notebook. If per-molecule tables are given in `DFT_FILES`, they are matched to the dataset by canonical SMILES (catalysts by canonical SMILES together with the axial configuration), mapped onto reactions in the order catalyst, electrophile, nucleophile, and written to `features_dft.csv` with the two catalyst axial flags, in the same layout as the other per-role representations. Every component of the dataset must be present in the corresponding table; missing structures are reported and stop the run. Descriptor values are written as given. If no table is supplied, the section is skipped and no file is written.

In [ ]:
def canonical(smi):
    return Chem.MolToSmiles(Chem.MolFromSmiles(smi))

if all(DFT_FILES[r] is None for r in ROLE_COLS):
    print('DFT tables not supplied - section skipped')
else:
    assert all(DFT_FILES[r] is not None for r in ROLE_COLS), 'Supply DFT tables for all three roles or for none'
    dft_blocks = []
    for role, col in ROLE_COLS.items():
        table = pd.read_csv(DFT_FILES[role], float_precision='round_trip')
        assert 'smiles' in table.columns, f'"smiles" column missing in {DFT_FILES[role]}'
        desc_cols = [c for c in table.columns if c not in ('smiles', AXIAL_COL)]
        non_numeric = [c for c in desc_cols if not pd.api.types.is_numeric_dtype(table[c])]
        assert not non_numeric, f'Non-numeric descriptor columns in {DFT_FILES[role]}: {non_numeric}'
        key = table['smiles'].map(canonical)
        if role == 'cat':
            assert AXIAL_COL in table.columns, f'"{AXIAL_COL}" column missing in {DFT_FILES[role]}'
            key = key + '|' + table[AXIAL_COL].fillna('none').astype(str).str.strip()
            wanted = pd.Series(cat_keys.index, index=cat_keys.index)
            wanted_key = cat_keys[col].map(canonical) + '|' + cat_keys[AXIAL_COL]
        else:
            wanted = pd.Series(unique_mols[role], index=unique_mols[role])
            wanted_key = wanted.map(canonical)
        assert key.is_unique, f'Duplicate structures in {DFT_FILES[role]}'
        lut = table.set_index(key)[desc_cols]
        missing = wanted.index[~wanted_key.isin(lut.index).values]
        assert len(missing) == 0, f'{len(missing)} {role} structures missing from DFT table, e.g. {list(missing[:5])}'
        lut = lut.loc[wanted_key.values].set_axis(wanted.index)
        lut.columns = [f'{role}_{c}' for c in lut.columns]
        if role == 'cat':
            lut = pd.concat([lut, axial_flags], axis=1)
            row_keys = df_cat_key
        else:
            row_keys = df[col]
        per_molecule[('dft', role)] = lut
        dft_blocks.append(lut.loc[row_keys].reset_index(drop=True))
    features['dft'] = pd.concat([ids] + dft_blocks, axis=1)
    print(f"dft          {features['dft'].shape[1] - 1:6d} columns")

## 8. Reaction conditions

Temperature (°C) and catalyst loading (mol %) are taken as given. The solvent is described by the Kamlet–Taft parameters α (hydrogen-bond donor acidity), β (hydrogen-bond acceptor basicity), π\* (dipolarity/polarisability) and the static dielectric constant ε (20–25 °C). Values follow the compilation of Marcus (*Chem. Soc. Rev.* **1993**, 22, 409); solvents absent from it take the values of the closest structural analogue, as stated in `source`. **The table must be checked against the primary sources before publication.** Solvent names are normalised by `SOLVENT_ALIASES` before lookup. The table of solvents used is written to `solvent_parameters.csv`.

In [ ]:
SOLVENT_ALIASES = {'DCM': 'CH2Cl2', 'PHCF3': 'PhCF3', 'TBME': 'MTBE', 'EVE': 'EtOCH=CH2',
                   'MeOH': 'CH3OH', 'MeCN': 'CH3CN'}

#  solvent        alpha  beta   pi*    eps    source
_SOLVENTS = [
    ('CH2Cl2',      0.13,  0.10,  0.82,  8.93, 'Marcus1993'),
    ('toluene',     0.00,  0.11,  0.54,  2.38, 'Marcus1993'),
    ('THF',         0.00,  0.55,  0.58,  7.58, 'Marcus1993'),
    ('Et2O',        0.00,  0.47,  0.27,  4.20, 'Marcus1993'),
    ('m-xylene',    0.00,  0.11,  0.47,  2.37, 'Marcus1993'),
    ('p-xylene',    0.00,  0.12,  0.43,  2.27, 'Marcus1993'),
    ('xylenes',     0.00,  0.11,  0.47,  2.37, 'analogue: m-xylene'),
    ('CH3CN',       0.19,  0.40,  0.75, 35.94, 'Marcus1993'),
    ('CHCl3',       0.20,  0.10,  0.58,  4.81, 'Marcus1993'),
    ('CPME',        0.00,  0.45,  0.25,  4.76, 'analogue: MTBE (alpha, beta, pi*); eps literature'),
    ('MTBE',        0.00,  0.45,  0.25,  2.60, 'Marcus1993'),
    ('acetone',     0.08,  0.43,  0.71, 20.70, 'Marcus1993'),
    ('dioxane',     0.00,  0.37,  0.55,  2.21, 'Marcus1993'),
    ('CCl4',        0.00,  0.10,  0.28,  2.24, 'Marcus1993'),
    ('DCE',         0.00,  0.10,  0.81, 10.36, 'Marcus1993'),
    ('EtOAc',       0.00,  0.45,  0.55,  6.02, 'Marcus1993'),
    ('DMF',         0.00,  0.69,  0.88, 36.70, 'Marcus1993'),
    ('DME',         0.00,  0.41,  0.53,  7.20, 'Marcus1993'),
    ('PhCF3',       0.00,  0.07,  0.71,  9.18, 'analogue: PhCl (alpha, beta, pi*); eps literature'),
    ('EtOH',        0.86,  0.75,  0.54, 24.55, 'Marcus1993'),
    ('i-PrOH',      0.76,  0.84,  0.48, 19.92, 'Marcus1993'),
    ('DMSO',        0.00,  0.76,  1.00, 46.70, 'Marcus1993'),
    ('hexane',      0.00,  0.00, -0.04,  1.88, 'Marcus1993'),
    ('CH3OH',       0.98,  0.66,  0.60, 32.66, 'Marcus1993'),
    ('Ph2O',        0.00,  0.13,  0.66,  3.60, 'Marcus1993'),
    ('EtOCH=CH2',   0.00,  0.47,  0.27,  3.40, 'analogue: Et2O (alpha, beta, pi*); eps estimate'),
    ('benzene',     0.00,  0.10,  0.59,  2.27, 'Marcus1993'),
    ('anisole',     0.00,  0.32,  0.73,  4.33, 'Marcus1993'),
    ('cyclohexane', 0.00,  0.00,  0.00,  2.02, 'Marcus1993'),
    ('PhCl',        0.00,  0.07,  0.71,  5.62, 'Marcus1993'),
]
solvent_table = pd.DataFrame(_SOLVENTS, columns=['solvent', 'alpha', 'beta', 'pi_star', 'eps', 'source'])
if SOLVENT_TABLE_PATH is not None:
    solvent_table = pd.read_csv(SOLVENT_TABLE_PATH)

solv = df[SOLVENT_COL].astype(str).str.strip().replace(SOLVENT_ALIASES)
unmatched = sorted(set(solv) - set(solvent_table.solvent))
assert not unmatched, f'Solvents without parameters: {unmatched}'

params = solvent_table.set_index('solvent').loc[solv, ['alpha', 'beta', 'pi_star', 'eps']].reset_index(drop=True)
conditions = pd.concat([ids,
                        pd.DataFrame({'cond_temperature_C': df[TEMP_COL].astype(float),
                                      'cond_loading_molpct': df[LOADING_COL].astype(float)}),
                        params.add_prefix('solv_')], axis=1)
assert not conditions.isna().any().any(), 'Missing values in reaction conditions'
features['conditions'] = conditions

used = solvent_table[solvent_table.solvent.isin(set(solv))].copy()
used['n_reactions'] = used.solvent.map(solv.value_counts())
used = used.sort_values('n_reactions', ascending=False).reset_index(drop=True)
used.to_csv(OUT_DIR / 'solvent_parameters.csv', index=False)
used

## 9. Representation diagnostics

For every representation: dimensionality, fraction of zero entries, number of columns constant over the dataset, missing values, and *resolution*, i.e. the number of distinct feature vectors among the unique molecules of each role (or among reactions). Two molecules with identical vectors cannot be distinguished by any model trained on that representation. For ECFP4, resolution is also reported without chirality and without the axial flags to quantify the contribution of each. The first row gives the number of unique structures as the upper bound: catalysts as (SMILES, axial configuration), reactions as unique combinations of catalyst, axial configuration, electrophile, nucleophile and product.

In [ ]:
def n_distinct(frame):
    return int(frame.round(10).astype(str).agg('|'.join, axis=1).nunique())

rows = []
for rep, frame in features.items():
    X = frame.drop(columns='reaction_id')
    row = {'Representation': rep, 'Columns': X.shape[1],
           'Zero fraction': float((X.to_numpy() == 0).mean()) if rep not in ('physchem', 'dft') else np.nan,
           'Constant columns': int((X.nunique(dropna=False) <= 1).sum()),
           'Columns with missing values': int(X.isna().any().sum()),
           'Distinct reaction vectors': n_distinct(X)}
    for role in ROLE_COLS:
        key = f'Distinct {role} vectors'
        row[key] = n_distinct(per_molecule[(rep, role)]) if (rep, role) in per_molecule else np.nan
    rows.append(row)
diag = pd.DataFrame(rows)
for role in ROLE_COLS:
    diag[f'Distinct {role} vectors'] = diag[f'Distinct {role} vectors'].astype('Int64')
unique_counts = pd.DataFrame([{'Representation': 'unique structures', 'Distinct reaction vectors':
                               len(df.drop_duplicates(list(ROLE_COLS.values()) + [AXIAL_COL, PRODUCT_COL])),
                               'Distinct cat vectors': len(cat_keys),
                               **{f'Distinct {r} vectors': len(unique_mols[r]) for r in ['ele', 'nuc']}}])
diag = pd.concat([unique_counts, diag], ignore_index=True)
diag = diag[['Representation', 'Columns', 'Zero fraction', 'Constant columns', 'Columns with missing values',
             'Distinct cat vectors', 'Distinct ele vectors', 'Distinct nuc vectors', 'Distinct reaction vectors']]
diag.to_csv(OUT_DIR / 'representation_diagnostics.csv', index=False)
diag

In [ ]:
achiral_gen = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_FP_SIZE,
                                                         includeChirality=False)
chirality_rows = []
for role in ROLE_COLS:
    smiles = unique_mols[role]
    achiral = pd.DataFrame(np.vstack([achiral_gen.GetFingerprintAsNumPy(mol_objects[s]) for s in smiles]))
    chiral = pd.DataFrame(np.vstack([MORGAN_GEN.GetFingerprintAsNumPy(mol_objects[s]) for s in smiles]))
    n_struct = len(cat_keys) if role == 'cat' else len(smiles)
    chirality_rows.append({'Role': role, 'Unique structures': n_struct, 'Unique SMILES': len(smiles),
                           'ECFP4, no chirality': n_distinct(achiral),
                           'ECFP4, includeChirality': n_distinct(chiral),
                           'ECFP4, includeChirality + axial flags': n_distinct(per_molecule[('ecfp4', role)]),
                           'Physchem + axial flags': n_distinct(per_molecule[('physchem', role)])})
chirality = pd.DataFrame(chirality_rows)
chirality.to_csv(OUT_DIR / 'representation_resolution_chirality.csv', index=False)
chirality

## 10. Output files and manifest

In [ ]:
FILES = {rep: OUT_DIR / f'features_{rep}.csv' for rep in features}
for rep, path in FILES.items():
    assert features[rep]['reaction_id'].equals(df['reaction_id'])
    features[rep].to_csv(path, index=False)

manifest = {
    'input': {'path': str(DATA_PATH), 'sha256': sha256(DATA_PATH), 'n_rows': len(df)},
    'columns': {'roles': ROLE_COLS, 'product': PRODUCT_COL, 'axial_configuration': AXIAL_COL, 'temperature': TEMP_COL,
                'loading': LOADING_COL, 'solvent': SOLVENT_COL},
    'parameters': {'morgan_radius': MORGAN_RADIUS, 'morgan_fp_size': MORGAN_FP_SIZE,
                   'morgan_include_chirality': MORGAN_CHIRALITY, 'drfp_fp_size': DRFP_FP_SIZE,
                   'drfp_radius': DRFP_RADIUS, 'drfp_input': 'electrophile.nucleophile>catalyst>product',
                   'physchem_excluded': ['Ipc'], 'maccs_bits': '1-166',
                   'catalyst_axial_flags': ['cat_axial_R', 'cat_axial_S'],
                   'solvent_table': str(SOLVENT_TABLE_PATH) if SOLVENT_TABLE_PATH else 'built-in (Marcus 1993)'},
    'onehot_legend': {'path': str(OUT_DIR / 'onehot_legend.csv'), 'sha256': sha256(OUT_DIR / 'onehot_legend.csv')},
    'dft_tables': {r: ({'path': str(p), 'sha256': sha256(p)} if p else None) for r, p in DFT_FILES.items()},
    'outputs': {rep: {'path': str(p), 'sha256': sha256(p), 'n_columns': features[rep].shape[1] - 1}
                for rep, p in FILES.items()},
    'versions': {'python': platform.python_version(), 'platform': platform.platform(),
                 'rdkit': rdkit.__version__, 'drfp': getattr(drfp, '__version__', 'unknown'),
                 'numpy': np.__version__, 'pandas': pd.__version__},
}
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
pd.DataFrame([{'File': p.name, 'Columns': features[r].shape[1] - 1, 'Rows': len(features[r]),
               'Size, MB': round(p.stat().st_size / 2**20, 1)} for r, p in FILES.items()])